# Grad-CAM and CAM variants

Class Activation Maps (CAMs) show which regions of an image most influenced the decision of a convolutional network. This notebook applies Grad-CAM to a ResNet50 pre-trained on ImageNet, compares it with other CAM methods, and shows how to explain a specific class.

Adapted from [Advanced AI explainability for PyTorch](https://github.com/jacobgil/pytorch-grad-cam).

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/06_detection-segmentation-visualization'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

In [ ]:
![ -f utils.py ] || gdown --id 1NUGtNIABwt4PSAkBXPcMtnKPPaZYmxP3 -O utils.py
![ -f chart.py ] || gdown --id 1Y8zhWoaLEkeiAIkpEQJL_FsQInc7i-Os -O chart.py

In [ ]:
%pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 19.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.4.8-py3-none-any.whl size=38244 sha256=949a84752c3c524003f90cfd6eb7f1e0bd125121cc612ebd052aa6401accb6a8
  Stored in directory: /root/.cache/pip/wheels/f8/04/36/94ff3c8a4215826a21946b34c01180817e606989fdf53f7cd6
Successfully built grad-cam


In [ ]:
from pytorch_grad_cam import GradCAM, HiResCAM, GradCAMPlusPlus, XGradCAM, LayerCAM, EigenCAM, ScoreCAM, AblationCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.io import read_image
from torchvision import transforms

import time

import matplotlib.pyplot as plt
import torch

from chart import show

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Resnet model
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(device)
model.eval()
preprocess = ResNet50_Weights.IMAGENET1K_V2.transforms() # applied
categories = ResNet50_Weights.IMAGENET1K_V2.meta["categories"]

# Target layer: last conv layer
target_layers = [model.layer4[-1]]

# Input image - https://github.com/EliSchwartz/imagenet-sample-images
# image = read_image("n01443537_goldfish.jpeg") # as a tensor [0, 255]
image = read_image("shitzu.jpeg") # as a tensor [0, 255]
# image = read_image("n02701002_ambulance.jpeg")
# image = read_image("dog_cat.jfif")
show(image.numpy().transpose([1, 2, 0]), scale=0.3)

## Grad-CAM

In [ ]:
center_crop = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
])


def compute_cam(cam_method, image, targets=None):
    """Return the CAM overlaid on the image and the time it took, in seconds."""
    batch = preprocess(image).unsqueeze(0).to(device) # channels first
    base_image = center_crop(image).numpy().transpose([1, 2, 0]) / 255.

    start = time.time()
    with cam_method(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=batch, targets=targets)[0, :]
    elapsed = time.time() - start

    return show_cam_on_image(base_image, grayscale_cam, use_rgb=True), elapsed


# If targets is None, the highest scoring class is explained
prediction = model(preprocess(image).unsqueeze(0).to(device)).squeeze(0).softmax(0)
class_id = prediction.argmax().item()
print(f"Prediction: {categories[class_id]} (class id={class_id}, probability={prediction[class_id]:.2f})")

visualization, _ = compute_cam(GradCAM, image)
show(visualization, scale=0.3)

## Comparing CAM methods

All these methods combine the activation maps of the target layer into a single heatmap. What changes is how the weight of each map is computed:

| Method | How the activation maps are weighted | Uses gradients? |
|---|---|---|
| Grad-CAM | Spatial average of the gradients of the class score | Yes |
| HiResCAM | Element-wise product of gradients and activations, without averaging | Yes |
| Grad-CAM++ | Weighted average of the gradients, using higher-order terms | Yes |
| XGradCAM | Gradients scaled by the normalized activations | Yes |
| LayerCAM | Positive gradients, kept at each spatial position | Yes |
| EigenCAM | First principal component of the activations (not class-specific) | No |
| ScoreCAM | Class score after masking the input with each activation map | No |
| AblationCAM | Drop in the class score when each activation map is removed | No |

ScoreCAM and AblationCAM need many forward passes, one group for each activation map, so they take much longer. Compare the times in the titles below (a GPU helps a lot here).

In [ ]:
CAM_METHODS = {
    "Grad-CAM": GradCAM,
    "HiResCAM": HiResCAM,
    "Grad-CAM++": GradCAMPlusPlus,
    "XGradCAM": XGradCAM,
    "LayerCAM": LayerCAM,
    "EigenCAM": EigenCAM,
    "ScoreCAM": ScoreCAM,
    "AblationCAM": AblationCAM,
}

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (name, cam_method) in zip(axes.ravel(), CAM_METHODS.items()):
    visualization, elapsed = compute_cam(cam_method, image)
    ax.imshow(visualization)
    ax.set_title(f"{name} ({elapsed:.1f} s)")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Explaining a specific class

By default, a CAM explains the highest scoring class. With `ClassifierOutputTarget` we can choose which class to explain, which is useful when the image contains more than one object. Below, the same image is explained for a dog class and for a cat class.

Class-specific methods such as Grad-CAM should highlight a different region for each class. EigenCAM ignores the target, so its map does not change.

In [ ]:
dog_cat = read_image("dog_cat.jfif")

DOG_CLASS, CAT_CLASS = 207, 281 # golden retriever, tabby cat
panels = [
    (GradCAM, DOG_CLASS),
    (GradCAM, CAT_CLASS),
    (EigenCAM, DOG_CLASS),
    (EigenCAM, CAT_CLASS),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (cam_method, class_id) in zip(axes, panels):
    visualization, _ = compute_cam(cam_method, dog_cat, targets=[ClassifierOutputTarget(class_id)])
    ax.imshow(visualization)
    ax.set_title(f"{cam_method.__name__}: {categories[class_id]}")
    ax.axis("off")
plt.tight_layout()
plt.show()